# Week 20 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [ ]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 20 Day 01: Market microstructure essentials

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Incorporate execution realism and microstructure awareness into a blended strategy portfolio.

## Continuity and Handoff
- Previous checkpoint: Week 19 Day 07: Portfolio Project
- Previous lesson file: content/week-19/day-07.md
- Today's deliverable: Create synthetic microstructure scenarios and expected slippage estimates.
- Next handoff target: Week 20 Day 02: Execution cost modeling
- Next lesson file: content/week-20/day-02.md

## Theory Concepts

### Concept 1: Order book dynamics
Order book dynamics should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Spread and depth
Spread and depth should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Latency and queue effects
Latency and queue effects should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: Cross-Sectional Z
$$
z_{i,t}=\frac{x_{i,t}-\mu_t}{\sigma_t}
$$
Universe-normalized signal.

### Formula 2: Information Coefficient
$$
IC_t=Corr(score_{i,t},r_{i,t+1})
$$
Signal/forward-return linkage.

### Formula 3: IC t-Statistic
$$
t_{IC}=\frac{\bar{IC}}{Std(IC)/\sqrt{T}}
$$
Signal persistence test.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $IC$: information coefficient
- $ADV$: average daily volume
- $IS$: implementation shortfall

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Discuss how spread/depth changes alter expected execution quality.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Market microstructure essentials'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create synthetic microstructure scenarios and expected slippage estimates.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
Which microstructure feature most impacts your strategy style?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(2001)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


# Week 20 Day 02: Execution cost modeling

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Incorporate execution realism and microstructure awareness into a blended strategy portfolio.

## Continuity and Handoff
- Previous checkpoint: Week 20 Day 01: Market microstructure essentials
- Previous lesson file: content/week-20/day-01.md
- Today's deliverable: Build cost model including spread and impact assumptions.
- Next handoff target: Week 20 Day 03: Strategy blending and diversification
- Next lesson file: content/week-20/day-03.md

## Theory Concepts

### Concept 1: Implementation shortfall
Implementation shortfall should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Impact models
Impact models should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Participation constraints
Participation constraints should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: Information Coefficient
$$
IC_t=Corr(score_{i,t},r_{i,t+1})
$$
Signal/forward-return linkage.

### Formula 2: IC t-Statistic
$$
t_{IC}=\frac{\bar{IC}}{Std(IC)/\sqrt{T}}
$$
Signal persistence test.

### Formula 3: Spread Z-Score
$$
z_t=\frac{s_t-\mu_s}{\sigma_s}
$$
Stat-arb entry normalization.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $IC$: information coefficient
- $ADV$: average daily volume
- $IS$: implementation shortfall

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Estimate implementation shortfall under different participation rates.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Execution cost modeling'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build cost model including spread and impact assumptions.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
What execution assumption is least defensible in your current model?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(2002)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


# Week 20 Day 03: Strategy blending and diversification

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Incorporate execution realism and microstructure awareness into a blended strategy portfolio.

## Continuity and Handoff
- Previous checkpoint: Week 20 Day 02: Execution cost modeling
- Previous lesson file: content/week-20/day-02.md
- Today's deliverable: Implement multi-sleeve blend with adjustable risk budgets.
- Next handoff target: Week 20 Day 04: Rebalancing and execution scheduling
- Next lesson file: content/week-20/day-04.md

## Theory Concepts

### Concept 1: Correlation across alpha sleeves
Correlation across alpha sleeves should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Risk budget allocation
Risk budget allocation should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Capacity constraints
Capacity constraints should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: IC t-Statistic
$$
t_{IC}=\frac{\bar{IC}}{Std(IC)/\sqrt{T}}
$$
Signal persistence test.

### Formula 2: Spread Z-Score
$$
z_t=\frac{s_t-\mu_s}{\sigma_s}
$$
Stat-arb entry normalization.

### Formula 3: Implementation Shortfall
$$
IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}
$$
Execution loss in bps.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $IC$: information coefficient
- $ADV$: average daily volume
- $IS$: implementation shortfall

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Blend momentum, mean-reversion, and factor sleeves under risk limits.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Strategy blending and diversification'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement multi-sleeve blend with adjustable risk budgets.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
How does capacity constraint change blend weights?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(2003)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


# Week 20 Day 04: Rebalancing and execution scheduling

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Incorporate execution realism and microstructure awareness into a blended strategy portfolio.

## Continuity and Handoff
- Previous checkpoint: Week 20 Day 03: Strategy blending and diversification
- Previous lesson file: content/week-20/day-03.md
- Today's deliverable: Create scheduler simulation for rebalance events.
- Next handoff target: Week 20 Day 05: Capstone assembly and governance
- Next lesson file: content/week-20/day-05.md

## Theory Concepts

### Concept 1: Trade scheduling
Trade scheduling should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Urgency vs cost tradeoff
Urgency vs cost tradeoff should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Execution windows
Execution windows should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: Spread Z-Score
$$
z_t=\frac{s_t-\mu_s}{\sigma_s}
$$
Stat-arb entry normalization.

### Formula 2: Implementation Shortfall
$$
IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}
$$
Execution loss in bps.

### Formula 3: Cross-Sectional Z
$$
z_{i,t}=\frac{x_{i,t}-\mu_t}{\sigma_t}
$$
Universe-normalized signal.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $IC$: information coefficient
- $ADV$: average daily volume
- $IS$: implementation shortfall

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Compare immediate vs scheduled execution impact assumptions.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Rebalancing and execution scheduling'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create scheduler simulation for rebalance events.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
When should urgency override cost minimization?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(2004)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


# Week 20 Day 05: Capstone assembly and governance

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Incorporate execution realism and microstructure awareness into a blended strategy portfolio.

## Continuity and Handoff
- Previous checkpoint: Week 20 Day 04: Rebalancing and execution scheduling
- Previous lesson file: content/week-20/day-04.md
- Today's deliverable: Generate final blend dashboard for capstone review.
- Next handoff target: Week 20 Day 06: Revision Sprint
- Next lesson file: content/week-20/day-06.md

## Theory Concepts

### Concept 1: Blend-level performance attribution
Blend-level performance attribution should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Execution-risk reporting
Execution-risk reporting should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Operational controls
Operational controls should be treated as a measurable component of 'Execution, microstructure, and multi-strategy blend'. For this week, emphasize alpha stability, execution realism, and risk-governed deployment. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: Implementation Shortfall
$$
IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}
$$
Execution loss in bps.

### Formula 2: Cross-Sectional Z
$$
z_{i,t}=\frac{x_{i,t}-\mu_t}{\sigma_t}
$$
Universe-normalized signal.

### Formula 3: Information Coefficient
$$
IC_t=Corr(score_{i,t},r_{i,t+1})
$$
Signal/forward-return linkage.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $IC$: information coefficient
- $ADV$: average daily volume
- $IS$: implementation shortfall

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Compile multi-strategy report with net-of-cost metrics.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Capstone assembly and governance'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Generate final blend dashboard for capstone review.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
Which sleeve contributes most net alpha after execution costs?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(2005)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


# Week 20 Day 06: Revision Sprint

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 20 Day 05: Capstone assembly and governance
- Previous lesson file: content/week-20/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 20 Day 07: Portfolio Project
- Next lesson file: content/week-20/day-07.md

## Revision Plan
- 30 minutes: active recall of weekday concepts
- 40 minutes: rebuild one code workflow from memory
- 30 minutes: error log triage and retest plan
- 20 minutes: update progress tracker and notes

## Focus Areas
- Re-check slippage assumptions against turnover
- Review blend risk-budget consistency
- Update execution governance checklist

## Revision Output
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Next-week risk list prepared


# Week 20 Day 07: Portfolio Project

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 20 Day 06: Revision Sprint
- Previous lesson file: content/week-20/day-06.md
- Today's deliverable: Capstone 5: Multi-strategy paper portfolio
- Next handoff target: Week 21 Day 01: Program targeting and fit matrix
- Next lesson file: content/week-21/day-01.md

## Project Title
Capstone 5: Multi-strategy paper portfolio

## Problem Statement
Construct a multi-sleeve paper portfolio with microstructure-aware execution assumptions.

## Data Sources
- Strategy sleeve signals
- Liquidity proxies
- Cost model inputs

## Implementation Steps
1. Design sleeve blend
2. Apply execution-cost assumptions
3. Backtest net performance
4. Analyze attribution
5. Produce governance report

## Evaluation Metrics
- Net Sharpe proxy
- Cost drag
- Capacity proxy
- Risk-budget adherence

## Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned
